# Task 1

Generate a table with the top 150 movies from IMDb

## 1. Libraries

In [ ]:
# !pip install -r requirements.txt

In [ ]:
import pandas as pd
import time

# Selenium tools
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException

## 2. Set driver

In [ ]:
chrome_options = Options()
chrome_options.add_argument("--start-maximized")
chrome_options.add_argument("--lang=en-US")

driver = webdriver.Chrome(options=chrome_options)
url = "https://www.imdb.com/chart/top/"
driver.get(url)

## 3. Scraping

In [ ]:
# Explicit wait
try:
    WebDriverWait(driver,10).until(
        EC.visibility_of_element_located((By.CSS_SELECTOR, "ul.ipc-metadata-list"))
    )
    print("Element found.")
except TimeoutException:
    print("Error. Script will be stopped.")
    driver.quit()

In [ ]:
# Loop
movies_data = []
movie_elements = driver.find_elements(By.CSS_SELECTOR, "li.ipc-metadata-list-summary-item")

for movie in movie_elements:
    try:
        # Rank and Title
        title_element = movie.find_element(By.CSS_SELECTOR, "h3.ipc-title__text")
        full_title_text = title_element.text
        rank, title = full_title_text.split('. ', 1)
        
        # Year, Length and Type
        metadata_elements = movie.find_elements(By.CSS_SELECTOR, "div.cli-title-metadata ul.ipc-inline-list > li")
        if len(metadata_elements) >= 3:
            year = metadata_elements[0].text           
            duration = metadata_elements[1].text       
            classification = metadata_elements[2].text
        
        # Rating
        rating_element = movie.find_element(By.CSS_SELECTOR, "span.ipc-rating-star")
        rating = rating_element.text 

        # Movie's URL
        url_element = movie.find_element(By.CSS_SELECTOR, "a.ipc-title-link-wrapper")
        movie_url = url_element.get_attribute('href')

        # Saving data
        movies_data.append({
            "Rank": rank,
            "Title": title,
            "Year": year,
            "Rating_IMDb": rating,
            "URL": movie_url
        })
        print(f"Scraped: #{rank} {title}")

    except Exception as e:
        print(f" Error extracting data. Error: {e}")
        continue

print(f"\nScraping completed. Data from {len(movies_data)} movies extracted.")